# Link to GitHub Repository

You can find it here: https://github.com/avajts/pic16b_project.git

# Overview

(1-3 paragraphs)

(a figure)

# Technical Component 1: Dynamic Website

(at least 2 paragraphs)

In [ ]:
# code snippet

(a figure)

# Technical Component 2: ___

(at least 2 paragraphs)

In [ ]:
# code snippet

(a figure)

# Technical Component 3: Machine Learning Model's Performance

The `K-Nearest Neighbors (KNN)`-based recommendation system demonstrates a strong ability to suggest songs that align with user preferences when trained on **genre-specific subsets** rather than the entire dataset. By first filtering the `Spotify dataset` by genre and then applying `KNN` to find the most similar tracks based on audio features such as *danceability*, *energy*, and *valence*, the model ensures that recommendations align with the user’s listening patterns. This genre-specific approach helps reduce the risk of recommendations being skewed toward dominant characteristics in the full dataset, which was observed when training the model without genre constraints. However, one limitation lies in the dataset itself—**genres in the Spotify dataset may not always be correctly labeled or well-represented**, leading to recommendations that may not fully capture a user’s actual taste. Additionally, certain genres, particularly newer ones like *indie* and *pop*, have fewer data points available, which reduces the model’s ability to generate high-quality recommendations due to a lack of sufficient training examples.

Another important consideration is the model's reliance on randomly selecting songs from the top recommendations. While randomness introduces diversity, it also increases the chance of selecting less relevant songs instead of the best possible matches. Furthermore, `KNN`'s performance heavily depends on the choice of distance metric (`Euclidean distance` in this case), which may not always capture musical similarity in an optimal way. A stronger approach might involve integrating `cosine similarity` or a hybrid recommendation system that combines `content-based filtering` with `collaborative filtering`. Lastly, the filtering mechanism ensures users are not recommended tracks they already have in their playlists, which enhances suggesting unfamiliar songs. Future improvements could involve *refining the dataset*, *experimenting with feature weighting*, or *incorporating deep learning models for more specific recommendations*.

```python
def recommend_songs(user_songs, spotify_tracks, genre=None, n=5, random_state=42):
    """
    Generate a personalized playlist of `n` songs from a specific genre based on the user's preferences.

    Args:
        df: The user's listening history with audio features.
        genre: The genre of songs to recommend.
        n: The number of songs to recommend (default is 5).
        random_state: Seed for reproducibility (default is 42).

    Returns:
        random_recommendations: A DataFrame containing the recommended songs.
    """
    # Filter the Spotify dataset to include only songs from the specified genre
    genre_tracks = spotify_tracks[spotify_tracks['track_genre'].str.lower() == genre.lower()]

    # Check if there are enough songs in the specified genre
    if len(genre_tracks) < n:
        raise ValueError(f"Not enough songs in the '{genre}' genre. Only {len(genre_tracks)} songs available.")

    # Select relevant features for the KNN model
    features = ['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
                'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

    # Prepare the feature matrix for the user's listening history
    X_user = user_songs[features]

    # Train the KNN model on the user's data
    # Standardize the features
    scaler = StandardScaler()
    X_user_scaled = scaler.fit_transform(X_user)

    # Ensure `n` does not exceed the number of available songs in the dataset
    n = min(n, len(X_user_scaled))

    # Train the KNN model on the user's features
    knn = NearestNeighbors(n_neighbors=n, metric='euclidean')  # Use Euclidean distance
    knn.fit(X_user_scaled)

    # Prepare the feature matrix for the genre-specific songs
    X_genre = genre_tracks[features]
    X_genre_scaled = scaler.transform(X_genre)

    # Find the nearest neighbors (most similar songs) in the genre-specific dataset
    distances, indices = knn.kneighbors(X_genre_scaled)

    # Flatten the indices array to get a list of all recommended song indices
    recommended_song_indices = indices.flatten()

    # Ensure indices are within the valid range of the genre_tracks DataFrame
    valid_indices = [idx for idx in recommended_song_indices if idx < len(genre_tracks)]

    if not valid_indices:
        raise ValueError("No valid recommendations found. Please check the input data.")

    # Get the recommended songs
    recommended_songs = genre_tracks.iloc[valid_indices].drop_duplicates(subset=['track_name', 'artists']).head(n)

    # Randomly select `n` songs from the recommendations
    random_recommendations = recommended_songs.sample(n=n, random_state=random_state)

    random_recommendations = random_recommendations[['artists', 'track_name', 'track_genre']]

    # Display the recommended songs
    return random_recommendations
```

The `correlation matrix heatmap` visually represents how different track features are related to each other, helping us identify patterns in the dataset that may influence song recommendations. The more positive a correlation is, the more strongly correlated the two features are, which implies we should use features like:
* energy and loudness
* valence and danceability
* speechiness and explicit

In [1]:
from eda import plot_correlation_heatmap
from music_rec import get_spotify_df

spotify_df = get_spotify_df()
plot_correlation_heatmap(spotify_df)

# Concluding Remarks

(include discussion of ethical ramifications)